# E-Learning Platform - Complete Data Seeding

This notebook seeds all the necessary data for the e-learning platform:
1. Create Tenant
2. Create Permission Scopes
3. Create Permissions
4. Create Roles
5. Assign Permissions to Roles
6. Create Users
7. Assign Roles to Users
8. Create Notes (Draft, Published, Released)
9. Create Mindmaps
10. Distribute Content to Classes

## Configuration

In [17]:
import requests
import json
import uuid
from datetime import datetime

# Server Configuration - UPDATE THIS
SERVER_IP = "192.168.0.105"  # Change to your server IP

# Service Ports
AUTH_PORT = 8081
NOTES_PORT = 8088
MINDMAP_PORT = 8087
WORKFLOW_PORT = 8086
ROLE_PERMISSION_PORT = 8080  # API Gateway
TENANT_PORT = 8080  # API Gateway

# Base URLs
AUTH_URL = f"http://{SERVER_IP}:{AUTH_PORT}"
NOTES_URL = f"http://{SERVER_IP}:{NOTES_PORT}"
MINDMAP_URL = f"http://{SERVER_IP}:{MINDMAP_PORT}"
WORKFLOW_URL = f"http://{SERVER_IP}:{WORKFLOW_PORT}"
ROLE_PERMISSION_URL = f"http://{SERVER_IP}:{ROLE_PERMISSION_PORT}"
TENANT_URL = f"http://{SERVER_IP}:{TENANT_PORT}"

# Tenant ID - Use a fixed UUID for consistency
TENANT_ID = "11111111-1111-1111-1111-111111111111"

# Store created IDs
created_data = {
    "tenant_id": TENANT_ID,
    "users": {},
    "roles": {},
    "permissions": {},
    "scopes": {},
    "notes": {},
    "mindmaps": {},
    "access_token": None
}

print(f"Server IP: {SERVER_IP}")
print(f"Tenant ID: {TENANT_ID}")
print(f"Auth URL: {AUTH_URL}")
print(f"Notes URL: {NOTES_URL}")
print(f"Role Permission URL: {ROLE_PERMISSION_URL}")

Server IP: 192.168.0.105
Tenant ID: 11111111-1111-1111-1111-111111111111
Auth URL: http://192.168.0.105:8081
Notes URL: http://192.168.0.105:8088
Role Permission URL: http://192.168.0.105:8080


## Helper Functions

In [18]:
def print_response(response, title="Response"):
    """Pretty print API response"""
    print(f"\n{'='*50}")
    print(f"{title}")
    print(f"Status: {response.status_code}")
    try:
        print(f"Body: {json.dumps(response.json(), indent=2)}")
    except:
        print(f"Body: {response.text[:500]}")
    print(f"{'='*50}\n")
    return response

def get_headers(with_auth=True, user_id="admin"):
    """Get common headers"""
    headers = {
        "Content-Type": "application/json",
        "X-Tenant-Id": TENANT_ID,
        "X-User-Id": user_id
    }
    if with_auth and created_data.get("access_token"):
        headers["Authorization"] = f"Bearer {created_data['access_token']}"
    return headers

## 1. Create Permission Scopes

In [19]:
# Permission scopes already exist in the system
# Just get existing scopes instead of creating
print("Getting existing Permission Scopes...")
try:
    response = requests.get(
        f"{ROLE_PERMISSION_URL}/permission-scopes",
        headers=get_headers(with_auth=False),
        timeout=10
    )
    if response.status_code == 200:
        scopes = response.json()
        for scope in scopes:
            created_data["scopes"][scope["code"]] = scope.get("id")
            print(f"  Found scope: {scope['code']}")
    else:
        print(f"Failed to get scopes: {response.status_code}")
except Exception as e:
    print(f"Error getting scopes: {e}")

print(f"\nScopes found: {list(created_data['scopes'].keys())}")

Getting existing Permission Scopes...
  Found scope: CLASS
  Found scope: TENANT
  Found scope: OWN
  Found scope: DEPARTMENT
  Found scope: PRIVATE
  Found scope: GROUP

Scopes found: ['CLASS', 'TENANT', 'OWN', 'DEPARTMENT', 'PRIVATE', 'GROUP']


## 2. Create Permissions

In [20]:
# Permissions already exist in the system
# Just get existing permissions instead of creating
print("Getting existing Permissions...")
try:
    response = requests.get(
        f"{ROLE_PERMISSION_URL}/permissions",
        headers=get_headers(with_auth=False),
        timeout=10
    )
    if response.status_code == 200:
        perms = response.json()
        for perm in perms:
            created_data["permissions"][perm["name"]] = perm.get("id")
            print(f"  Found permission: {perm['name']}")
    else:
        print(f"Failed to get permissions: {response.status_code}")
except Exception as e:
    print(f"Error getting permissions: {e}")

print(f"\nPermissions found: {len(created_data['permissions'])}")

Getting existing Permissions...
  Found permission: NOTE:READ
  Found permission: Create Notes
  Found permission: Read Notes
  Found permission: Update Notes
  Found permission: Delete Notes
  Found permission: NOTE:CREATE
  Found permission: NOTE:UPDATE
  Found permission: NOTE:DELETE
  Found permission: NOTE:PUBLISH
  Found permission: MINDMAP:CREATE
  Found permission: MINDMAP:READ
  Found permission: MINDMAP:UPDATE
  Found permission: MINDMAP:DELETE
  Found permission: WORKFLOW:CREATE
  Found permission: WORKFLOW:REVIEW
  Found permission: WORKFLOW:APPROVE
  Found permission: WORKFLOW:PUBLISH
  Found permission: CLASS:READ
  Found permission: CLASS:MANAGE
  Found permission: USER:READ
  Found permission: USER:MANAGE

Permissions found: 21


## 3. Create Roles

In [21]:
# Roles already exist in the system
# Just get existing roles instead of creating
print("Getting existing Roles...")
try:
    response = requests.get(
        f"{ROLE_PERMISSION_URL}/tenants/{TENANT_ID}/roles",
        headers=get_headers(with_auth=False),
        timeout=10
    )
    if response.status_code == 200:
        roles = response.json()
        for role in roles:
            created_data["roles"][role["name"]] = role.get("id")
            print(f"  Found role: {role['name']} (ID: {role.get('id')})")
    else:
        print(f"Failed to get roles: {response.status_code}")
except Exception as e:
    print(f"Error getting roles: {e}")

print(f"\nRoles found: {created_data['roles']}")

Getting existing Roles...
  Found role: ADMIN (ID: 11)
  Found role: TEACHER (ID: 12)
  Found role: REVIEWER (ID: 13)
  Found role: STUDENT (ID: 14)

Roles found: {'ADMIN': 11, 'TEACHER': 12, 'REVIEWER': 13, 'STUDENT': 14}


## 4. Assign Permissions to Roles (Role Grants)

In [22]:
# Define role-permission mappings
role_permissions = {
    "ADMIN": [
        "NOTE:CREATE", "NOTE:READ", "NOTE:UPDATE", "NOTE:DELETE", "NOTE:PUBLISH",
        "MINDMAP:CREATE", "MINDMAP:READ", "MINDMAP:UPDATE", "MINDMAP:DELETE",
        "WORKFLOW:CREATE", "WORKFLOW:REVIEW", "WORKFLOW:APPROVE", "WORKFLOW:PUBLISH",
        "CLASS:READ", "CLASS:MANAGE", "USER:READ", "USER:MANAGE"
    ],
    "TEACHER": [
        "NOTE:CREATE", "NOTE:READ", "NOTE:UPDATE", "NOTE:DELETE", "NOTE:PUBLISH",
        "MINDMAP:CREATE", "MINDMAP:READ", "MINDMAP:UPDATE", "MINDMAP:DELETE",
        "WORKFLOW:CREATE", "WORKFLOW:PUBLISH",
        "CLASS:READ"
    ],
    "REVIEWER": [
        "NOTE:READ", "MINDMAP:READ",
        "WORKFLOW:REVIEW", "WORKFLOW:APPROVE"
    ],
    "STUDENT": [
        "NOTE:READ", "MINDMAP:READ", "CLASS:READ"
    ]
}

print("Checking/Assigning Permissions to Roles...")
for role_name, permissions in role_permissions.items():
    role_id = created_data["roles"].get(role_name)
    if not role_id:
        print(f"  Skipping {role_name} - role not found")
        continue
    
    # Get existing grants for this role
    try:
        existing_response = requests.get(
            f"{ROLE_PERMISSION_URL}/tenants/{TENANT_ID}/roles/{role_id}/grants",
            headers=get_headers(with_auth=False),
            timeout=10
        )
        existing_grants = set()
        if existing_response.status_code == 200:
            for grant in existing_response.json():
                existing_grants.add(grant.get("permissionCode"))
    except:
        existing_grants = set()
    
    for perm_code in permissions:
        if perm_code in existing_grants:
            print(f"  Already granted: {perm_code} -> {role_name}")
            continue
            
        try:
            grant_data = {
                "permissionCode": perm_code,
                "scopeCode": "TENANT",
                "constraintsJson": None
            }
            response = requests.post(
                f"{ROLE_PERMISSION_URL}/tenants/{TENANT_ID}/roles/{role_id}/grants",
                headers=get_headers(with_auth=False),
                json=grant_data,
                timeout=10
            )
            if response.status_code in [200, 201]:
                print(f"  Granted {perm_code} to {role_name}")
            elif response.status_code == 409:
                print(f"  Already exists: {perm_code} -> {role_name}")
            else:
                print(f"  Failed: {perm_code} -> {role_name}: {response.status_code}")
        except Exception as e:
            print(f"  Error: {perm_code} -> {role_name}: {e}")

print("\nRole permission grants completed!")

Checking/Assigning Permissions to Roles...
  Already granted: NOTE:CREATE -> ADMIN
  Already granted: NOTE:READ -> ADMIN
  Already granted: NOTE:UPDATE -> ADMIN
  Already granted: NOTE:DELETE -> ADMIN
  Already granted: NOTE:PUBLISH -> ADMIN
  Already granted: MINDMAP:CREATE -> ADMIN
  Already granted: MINDMAP:READ -> ADMIN
  Already granted: MINDMAP:UPDATE -> ADMIN
  Already granted: MINDMAP:DELETE -> ADMIN
  Already granted: WORKFLOW:CREATE -> ADMIN
  Already granted: WORKFLOW:REVIEW -> ADMIN
  Already granted: WORKFLOW:APPROVE -> ADMIN
  Already granted: WORKFLOW:PUBLISH -> ADMIN
  Already granted: CLASS:READ -> ADMIN
  Already granted: CLASS:MANAGE -> ADMIN
  Already granted: USER:READ -> ADMIN
  Already granted: USER:MANAGE -> ADMIN
  Already granted: NOTE:CREATE -> TEACHER
  Already granted: NOTE:READ -> TEACHER
  Already granted: NOTE:UPDATE -> TEACHER
  Already granted: NOTE:DELETE -> TEACHER
  Already granted: NOTE:PUBLISH -> TEACHER
  Already granted: MINDMAP:CREATE -> TEACHE

## 5. Create Users

In [23]:
# Define users
users = [
    # Admin
    {"email": "admin@school.com", "password": "Admin@123", "name": "System Admin", "phone": "+919000000001", "role": "ADMIN"},
    
    # Teachers
    {"email": "john.smith@school.com", "password": "Teacher@123", "name": "John Smith", "phone": "+919000000002", "role": "TEACHER"},
    {"email": "sarah.johnson@school.com", "password": "Teacher@123", "name": "Sarah Johnson", "phone": "+919000000003", "role": "TEACHER"},
    {"email": "michael.brown@school.com", "password": "Teacher@123", "name": "Michael Brown", "phone": "+919000000004", "role": "TEACHER"},
    
    # Reviewers
    {"email": "emily.davis@school.com", "password": "Reviewer@123", "name": "Emily Davis", "phone": "+919000000005", "role": "REVIEWER"},
    {"email": "david.wilson@school.com", "password": "Reviewer@123", "name": "David Wilson", "phone": "+919000000006", "role": "REVIEWER"},
    
    # Students
    {"email": "student1@school.com", "password": "Student@123", "name": "Alice Anderson", "phone": "+919000000010", "role": "STUDENT"},
    {"email": "student2@school.com", "password": "Student@123", "name": "Bob Baker", "phone": "+919000000011", "role": "STUDENT"},
    {"email": "student3@school.com", "password": "Student@123", "name": "Carol Clark", "phone": "+919000000012", "role": "STUDENT"},
    {"email": "student4@school.com", "password": "Student@123", "name": "Daniel Drake", "phone": "+919000000013", "role": "STUDENT"},
    {"email": "student5@school.com", "password": "Student@123", "name": "Eva Evans", "phone": "+919000000014", "role": "STUDENT"}
]

print("Creating Users...")
for user in users:
    try:
        signup_data = {
            "tenantId": TENANT_ID,
            "email": user["email"],
            "phone": user["phone"],
            "password": user["password"],
            "name": user["name"],
            "joinMethod": "SELF_SIGNUP"
        }
        response = requests.post(
            f"{AUTH_URL}/auth/signup",
            headers={"Content-Type": "application/json"},
            json=signup_data,
            timeout=10
        )
        if response.status_code in [200, 201]:
            data = response.json()
            user_id = data.get("userId") or data.get("id")
            created_data["users"][user["email"]] = {
                "id": user_id,
                "role": user["role"],
                "password": user["password"]
            }
            print(f"  Created user: {user['name']} ({user['email']}) - Role: {user['role']}")
        elif response.status_code == 409 or "already exists" in response.text.lower():
            print(f"  User already exists: {user['email']}")
            created_data["users"][user["email"]] = {"role": user["role"], "password": user["password"]}
        else:
            print(f"  Failed to create user {user['email']}: {response.status_code} - {response.text[:100]}")
    except Exception as e:
        print(f"  Error creating user {user['email']}: {e}")

print(f"\nUsers created: {len(created_data['users'])}")

Creating Users...
  Created user: System Admin (admin@school.com) - Role: ADMIN
  Created user: John Smith (john.smith@school.com) - Role: TEACHER
  Created user: Sarah Johnson (sarah.johnson@school.com) - Role: TEACHER
  Created user: Michael Brown (michael.brown@school.com) - Role: TEACHER
  Created user: Emily Davis (emily.davis@school.com) - Role: REVIEWER
  Created user: David Wilson (david.wilson@school.com) - Role: REVIEWER
  Created user: Alice Anderson (student1@school.com) - Role: STUDENT
  Created user: Bob Baker (student2@school.com) - Role: STUDENT
  Created user: Carol Clark (student3@school.com) - Role: STUDENT
  Created user: Daniel Drake (student4@school.com) - Role: STUDENT
  Created user: Eva Evans (student5@school.com) - Role: STUDENT

Users created: 11


## 6. Login as Admin to Get Token

In [24]:
# Login as admin to get access token
print("Logging in as admin...")
try:
    login_data = {
        "tenantId": TENANT_ID,
        "identifier": "admin@school.com",
        "password": "Admin@123",
        "otp": ""
    }
    response = requests.post(
        f"{AUTH_URL}/auth/login",
        headers={"Content-Type": "application/json"},
        json=login_data,
        timeout=10
    )
    if response.status_code == 200:
        data = response.json()
        created_data["access_token"] = data.get("accessToken") or data.get("access_token") or data.get("token")
        admin_user_id = data.get("userId")
        if admin_user_id and "admin@school.com" in created_data["users"]:
            created_data["users"]["admin@school.com"]["id"] = admin_user_id
        print(f"  Login successful! Token: {created_data['access_token'][:50]}...")
    else:
        print(f"  Login failed: {response.status_code} - {response.text}")
except Exception as e:
    print(f"  Error logging in: {e}")

Logging in as admin...
  Login successful! Token: 1a587713-906c-4076-b769-7ab34b6a38e0...


## 7. Assign Roles to Users

In [25]:
print("Assigning Roles to Users...")
for email, user_data in created_data["users"].items():
    user_id = user_data.get("id")
    role_name = user_data.get("role")
    role_id = created_data["roles"].get(role_name)
    
    if not user_id:
        print(f"  Skipping {email} - user ID not found")
        continue
    if not role_id:
        print(f"  Skipping {email} - role {role_name} not found")
        continue
    
    try:
        assignment_data = {
            "roleId": role_id,
            "scopeType": "TENANT",
            "scopeId": None,
            "status": "ACTIVE"
        }
        response = requests.post(
            f"{ROLE_PERMISSION_URL}/tenants/{TENANT_ID}/users/{user_id}/roles",
            headers=get_headers(),
            json=assignment_data,
            timeout=10
        )
        if response.status_code in [200, 201]:
            print(f"  Assigned {role_name} to {email}")
        elif response.status_code == 409:
            print(f"  Role already assigned: {role_name} -> {email}")
        else:
            print(f"  Failed to assign role to {email}: {response.status_code}")
    except Exception as e:
        print(f"  Error assigning role to {email}: {e}")

print("\nRole assignments completed!")

Assigning Roles to Users...
  Assigned ADMIN to admin@school.com
  Assigned TEACHER to john.smith@school.com
  Assigned TEACHER to sarah.johnson@school.com
  Assigned TEACHER to michael.brown@school.com
  Assigned REVIEWER to emily.davis@school.com
  Assigned REVIEWER to david.wilson@school.com
  Assigned STUDENT to student1@school.com
  Assigned STUDENT to student2@school.com
  Assigned STUDENT to student3@school.com
  Assigned STUDENT to student4@school.com
  Assigned STUDENT to student5@school.com

Role assignments completed!


## 8. Login as Teacher to Create Content

In [26]:
# Login as teacher
print("Logging in as teacher (john.smith@school.com)...")
try:
    login_data = {
        "tenantId": TENANT_ID,
        "identifier": "john.smith@school.com",
        "password": "Teacher@123",
        "otp": ""
    }
    response = requests.post(
        f"{AUTH_URL}/auth/login",
        headers={"Content-Type": "application/json"},
        json=login_data,
        timeout=10
    )
    if response.status_code == 200:
        data = response.json()
        created_data["access_token"] = data.get("accessToken") or data.get("access_token")
        teacher_user_id = data.get("userId")
        created_data["teacher_user_id"] = teacher_user_id
        print(f"  Login successful! User ID: {teacher_user_id}")
    else:
        print(f"  Login failed: {response.status_code}")
except Exception as e:
    print(f"  Error: {e}")

Logging in as teacher (john.smith@school.com)...
  Login successful! User ID: a2c19d4e-6396-4c6a-9dd1-c2b9d58a19c8


## 9. Create Notes with Different Statuses

In [27]:
teacher_id = created_data.get("teacher_user_id", "teacher-001")

# Notes - all created as DRAFT first
# Publishing happens via workflow service (see next section)
notes = [
    {"title": "Introduction to Quadratic Equations", "summary": "Understanding quadratic equations", 
     "contentMd": "# Quadratic Equations\n\n$ax^2 + bx + c = 0$", "tags": ["math", "algebra"], "_publish": False},
    {"title": "Cell Division - Mitosis", "summary": "Cell division through mitosis",
     "contentMd": "# Mitosis\n\nPhases: Prophase, Metaphase, Anaphase, Telophase", "tags": ["biology"], "_publish": False},
    {"title": "French Revolution Timeline", "summary": "Key events",
     "contentMd": "# French Revolution\n\n1789-1799", "tags": ["history"], "_publish": False},
    {"title": "Newton's Laws of Motion", "summary": "Laws of classical mechanics",
     "contentMd": "# Newton's Laws\n\nF = ma", "tags": ["physics"], "_publish": True},
    {"title": "Periodic Table Introduction", "summary": "Elements organization",
     "contentMd": "# Periodic Table\n\nGroups and Periods", "tags": ["chemistry"], "_publish": True},
    {"title": "Shakespeare's Macbeth", "summary": "Famous tragedy summary",
     "contentMd": "# Macbeth\n\nAmbition and guilt", "tags": ["english", "literature"], "_publish": True},
    {"title": "Python Programming Basics", "summary": "Getting started with Python",
     "contentMd": "# Python\n\nprint('Hello World')", "tags": ["programming"], "_publish": False},
    {"title": "Human Digestive System", "summary": "How digestion works",
     "contentMd": "# Digestion\n\nMouth to intestines", "tags": ["biology", "anatomy"], "_publish": False},
]

print("Creating Notes...")
for note in notes:
    should_publish = note.pop("_publish", False)
    note_data = {
        "tenantId": TENANT_ID,
        "title": note["title"],
        "summary": note["summary"],
        "contentMd": note["contentMd"],
        "changeSummary": "Initial version",
        "tags": note["tags"],
        "scopeType": "TENANT",
        "createdBy": teacher_id
    }
    try:
        response = requests.post(
            f"{NOTES_URL}/notes",
            headers=get_headers(user_id=teacher_id),
            json=note_data,
            timeout=10
        )
        if response.status_code in [200, 201]:
            data = response.json()
            note_id = data.get("id")
            version_id = data.get("latestVersionId")
            created_data["notes"][note["title"]] = {
                "id": note_id,
                "versionId": version_id,
                "publish": should_publish
            }
            status = "TO_PUBLISH" if should_publish else "DRAFT"
            print(f"  Created: {note['title']} ({status})")
        else:
            print(f"  Failed: {note['title']} - {response.status_code}")
    except Exception as e:
        print(f"  Error: {note['title']} - {e}")

print(f"\nNotes created: {len(created_data['notes'])}")
to_publish = sum(1 for n in created_data["notes"].values() if n.get("publish"))
print(f"Notes to publish via workflow: {to_publish}")

Creating Notes...
  Created: Introduction to Quadratic Equations (DRAFT)
  Created: Cell Division - Mitosis (DRAFT)
  Created: French Revolution Timeline (DRAFT)
  Created: Newton's Laws of Motion (TO_PUBLISH)
  Created: Periodic Table Introduction (TO_PUBLISH)
  Created: Shakespeare's Macbeth (TO_PUBLISH)
  Created: Python Programming Basics (DRAFT)
  Created: Human Digestive System (DRAFT)

Notes created: 8
Notes to publish via workflow: 3


## 10. Create Mindmaps

In [28]:
# Mindmaps
mindmaps = [
    {
        "title": "Solar System",
        "subject": "Science",
        "grade": "8",
        "tags": ["astronomy", "planets", "space"],
        "visibility": "TENANT"
    },
    {
        "title": "Parts of Speech",
        "subject": "English",
        "grade": "7",
        "tags": ["grammar", "language"],
        "visibility": "TENANT"
    },
    {
        "title": "Human Body Systems",
        "subject": "Biology",
        "grade": "9",
        "tags": ["anatomy", "biology"],
        "visibility": "TENANT"
    },
    {
        "title": "Mathematical Operations",
        "subject": "Math",
        "grade": "6",
        "tags": ["arithmetic", "basics"],
        "visibility": "TENANT"
    },
    {
        "title": "World War I Causes",
        "subject": "History",
        "grade": "10",
        "tags": ["history", "wwi"],
        "visibility": "TENANT"
    }
]

print("Creating Mindmaps...")
for mindmap in mindmaps:
    try:
        response = requests.post(
            f"{MINDMAP_URL}/mindmaps",
            headers=get_headers(user_id=teacher_id),
            json=mindmap,
            timeout=10
        )
        if response.status_code in [200, 201]:
            data = response.json()
            created_data["mindmaps"][mindmap["title"]] = data.get("mindMapId")
            print(f"  Created mindmap: {mindmap['title']}")
        else:
            print(f"  Failed to create mindmap {mindmap['title']}: {response.status_code}")
    except Exception as e:
        print(f"  Error creating mindmap {mindmap['title']}: {e}")

print(f"\nMindmaps created: {len(created_data['mindmaps'])}")

Creating Mindmaps...
  Created mindmap: Solar System
  Created mindmap: Parts of Speech
  Created mindmap: Human Body Systems
  Created mindmap: Mathematical Operations
  Created mindmap: World War I Causes

Mindmaps created: 5


## 11. Create Workflow (Distribute Content to Classes)

In [29]:
# Create workflows for notes marked for publishing
# Workflow lifecycle: DRAFT -> IN_REVIEW -> APPROVED -> PUBLISHED
print("Creating Workflows and Publishing Content...")

class_ids = ["class_8A", "class_8B", "class_9A", "class_10A"]
teacher_id = created_data.get("teacher_user_id")

for title, note_data in created_data["notes"].items():
    if note_data.get("publish"):
        content_id = note_data["id"]
        
        # Check if workflow already exists for this content
        check_resp = requests.get(
            f"{WORKFLOW_URL}/workflow?contentId={content_id}&size=10",
            headers=get_headers(user_id=teacher_id),
            timeout=10
        )
        existing_workflow = None
        if check_resp.status_code == 200:
            data = check_resp.json()
            workflows = data.get("content", data.get("items", []))
            if workflows:
                existing_workflow = workflows[0]
                workflow_id = existing_workflow.get("workflowId")
                state = existing_workflow.get("state")
                print(f"  {title}: Existing workflow found (state: {state})")
                
                if state == "PUBLISHED":
                    print(f"    -> Already published, skipping")
                    continue
                elif state == "APPROVED":
                    # Just publish
                    r = requests.post(f"{WORKFLOW_URL}/workflow/{workflow_id}/publish", 
                                      headers=get_headers(user_id=teacher_id),
                                      json={"publishAt": None, "publishTargets": {"scope": "CLASS", "classIds": class_ids[:2]}}, 
                                      timeout=10)
                    if r.status_code in [200, 201]:
                        print(f"    -> PUBLISHED!")
                    else:
                        print(f"    -> Publish failed: {r.status_code}")
                    continue
        
        try:
            if not existing_workflow:
                # Step 1: Create new workflow
                workflow_data = {
                    "contentId": content_id,
                    "contentVersionId": note_data.get("versionId", "v1"),
                    "titleSnapshot": title,
                    "publishTargets": {"scope": "CLASS", "classIds": class_ids[:2]}
                }
                response = requests.post(
                    f"{WORKFLOW_URL}/workflow",
                    headers=get_headers(user_id=teacher_id),
                    json=workflow_data,
                    timeout=10
                )
                if response.status_code not in [200, 201]:
                    print(f"  {title}: Failed to create workflow ({response.status_code})")
                    continue
                    
                wf = response.json()
                workflow_id = wf.get("workflowId")
                print(f"  {title}: Created new workflow")
            
            # Step 2: Submit for review
            submit_data = {"reviewerUserIds": [teacher_id], "requiredApprovals": 1, "note": "Auto-review"}
            r = requests.post(f"{WORKFLOW_URL}/workflow/{workflow_id}/submit", 
                              headers=get_headers(user_id=teacher_id), json=submit_data, timeout=10)
            if r.status_code not in [200, 201]:
                print(f"    -> Submit failed: {r.status_code}")
                continue
            print(f"    -> Submitted")
            
            # Step 3: Get review task ID
            r = requests.get(f"{WORKFLOW_URL}/workflow/{workflow_id}", headers=get_headers(user_id=teacher_id), timeout=10)
            if r.status_code == 200:
                wf = r.json()
                reviews = wf.get("reviewTasks", [])
                if reviews:
                    review_task_id = reviews[0].get("id")
                    
                    # Step 4: Approve
                    r = requests.post(f"{WORKFLOW_URL}/workflow/{workflow_id}/reviews/{review_task_id}/approve", 
                                      headers=get_headers(user_id=teacher_id), json={"comment": "Auto-approved"}, timeout=10)
                    if r.status_code in [200, 201]:
                        print(f"    -> Approved")
                        
                        # Step 5: Publish
                        r = requests.post(f"{WORKFLOW_URL}/workflow/{workflow_id}/publish", 
                                          headers=get_headers(user_id=teacher_id),
                                          json={"publishAt": None, "publishTargets": {"scope": "CLASS", "classIds": class_ids[:2]}}, 
                                          timeout=10)
                        if r.status_code in [200, 201]:
                            print(f"    -> PUBLISHED!")
                        else:
                            print(f"    -> Publish failed: {r.status_code}")
                    else:
                        print(f"    -> Approve failed: {r.status_code}")
        except Exception as e:
            print(f"  Error for {title}: {e}")

print("\nWorkflow distribution completed!")

Creating Workflows and Publishing Content...
  Newton's Laws of Motion: Created new workflow
    -> Submitted
    -> Approved
    -> PUBLISHED!
  Periodic Table Introduction: Created new workflow
    -> Submitted
    -> Approved
    -> PUBLISHED!
  Shakespeare's Macbeth: Created new workflow
    -> Submitted
    -> Approved
    -> PUBLISHED!

Workflow distribution completed!


## Summary

In [30]:
print("\n" + "="*60)
print("SEEDING COMPLETE!")
print("="*60)

print(f"\nTenant ID: {TENANT_ID}")
print(f"Server IP: {SERVER_IP}")

print(f"\nScopes created: {len(created_data['scopes'])}")
print(f"Permissions created: {len(created_data['permissions'])}")
print(f"Roles created: {len(created_data['roles'])}")
print(f"Users created: {len(created_data['users'])}")
print(f"Notes created: {len(created_data['notes'])}")
print(f"Mindmaps created: {len(created_data['mindmaps'])}")

print("\n" + "-"*60)
print("USER CREDENTIALS:")
print("-"*60)
print(f"{'Email':<35} {'Password':<15} {'Role'}")
print("-"*60)
for email, user_data in created_data["users"].items():
    print(f"{email:<35} {user_data.get('password', 'N/A'):<15} {user_data.get('role', 'N/A')}")

print("\n" + "-"*60)
print("NOTES STATUS:")
print("-"*60)
draft_count = sum(1 for n in created_data["notes"].values() if n.get("status") == "DRAFT")
published_count = sum(1 for n in created_data["notes"].values() if n.get("status") == "PUBLISHED")
print(f"Draft: {draft_count}")
print(f"Published: {published_count}")

print("\n" + "="*60)
print("You can now use the Flutter app with these credentials!")
print("="*60)


SEEDING COMPLETE!

Tenant ID: 11111111-1111-1111-1111-111111111111
Server IP: 192.168.0.105

Scopes created: 6
Permissions created: 21
Roles created: 4
Users created: 11
Notes created: 8
Mindmaps created: 5

------------------------------------------------------------
USER CREDENTIALS:
------------------------------------------------------------
Email                               Password        Role
------------------------------------------------------------
admin@school.com                    Admin@123       ADMIN
john.smith@school.com               Teacher@123     TEACHER
sarah.johnson@school.com            Teacher@123     TEACHER
michael.brown@school.com            Teacher@123     TEACHER
emily.davis@school.com              Reviewer@123    REVIEWER
david.wilson@school.com             Reviewer@123    REVIEWER
student1@school.com                 Student@123     STUDENT
student2@school.com                 Student@123     STUDENT
student3@school.com                 Student@123     S


Email	Password	Role
teacher1@school.com	Teacher@123	TEACHER
teacher2@school.com	Teacher@123	TEACHER
admin@school.com	Admin@123	ADMIN
reviewer1@school.com	Reviewer@123	REVIEWER
student1@school.com	Student@123	STUDENT
Use teacher1@school.com / Teacher@123